# TruthLens UA — Ukrainian NLP Training & A/B Testing
Priority: WUNU UA Corpus + UNLP 2025 Telegram. A/B test LinearSVC, LogReg, RandomForest.
Student: 102012dl

In [ ]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings("ignore")
print("Imports OK")

In [ ]:
# Load UA/EN data: UNLP 2025 (HuggingFace) or manual UA samples + ISOT if available
def load_ua_data():
    try:
        from datasets import load_dataset
        ds = load_dataset("ukr-detect/ukr-twitter-2021", trust_remote_code=True)
        df = ds["train"].to_pandas()
        df = df.rename(columns={"tweet": "text"}) if "tweet" in df.columns else df
        df["label"] = df["label"].map({0: "REAL", 1: "FAKE"})
        return df["text"].astype(str), df["label"]
    except Exception as e:
        print("UNLP load failed:", e)
    # Manual UA samples for demo
    texts = [
        "ТЕРМІНОВО!!! ЗСУ ЗДАЛИ Харків! Поширте до видалення!!!",
        "НБУ підвищив облікову ставку до 16% річних.",
        "Уряд ПРИХОВУЄ правду! Прокидайтесь люди!!!",
        "Кабінет Міністрів затвердив програму підтримки бізнесу.",
    ] * 25
    labels = ["FAKE", "REAL", "FAKE", "REAL"] * 25
    return pd.Series(texts), pd.Series(labels)

X_ua, y_ua = load_ua_data()
X_ua, y_ua = X_ua.tolist(), y_ua.tolist()
print(f"UA samples: {len(X_ua)}", pd.Series(y_ua).value_counts().to_dict())

In [ ]:
# Optional: merge with ISOT for larger train set (run 01 first to have data)
try:
    fake = pd.read_csv("data/Fake.csv")
    true = pd.read_csv("data/True.csv")
    text_col = "text" if "text" in fake.columns else "title"
    fake["label"], true["label"] = "FAKE", "REAL"
    iso = pd.concat([fake[[text_col, "label"]], true[[text_col, "label"]]])
    iso = iso.dropna(subset=[text_col])
    X_iso = iso[text_col].astype(str).tolist()
    y_iso = iso["label"].tolist()
    X_all = X_ua + X_iso
    y_all = y_ua + y_iso
    print(f"Merged: {len(X_all)} total")
except Exception as e:
    print("ISOT merge skip:", e)
    X_all, y_all = X_ua, y_ua

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
mlflow.set_experiment("ab-test-ukraine-nlp")

models = {
    "LinearSVC_C1": LinearSVC(max_iter=10000, C=1.0),
    "LinearSVC_C05": LinearSVC(max_iter=10000, C=0.5),
    "LogisticRegression": LogisticRegression(max_iter=1000, C=1.0),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
}
results = {}
for name, clf in models.items():
    with mlflow.start_run(run_name=f"AB_{name}"):
        mlflow.set_tags({"ab_test": True, "student": "102012dl", "language": "ua+en"})
        pipe = Pipeline([
            ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2)),
            ("clf", clf),
        ])
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)
        f1 = f1_score(y_test, preds, average="weighted")
        acc = accuracy_score(y_test, preds)
        mlflow.log_metrics({"f1": f1, "accuracy": acc})
        mlflow.sklearn.log_model(pipe, f"model_{name}")
        results[name] = {"f1": f1, "acc": acc}
        print(f"{name}: F1={f1:.4f} Acc={acc:.4f}")
best = max(results, key=lambda k: results[k]["f1"])
print(f"Winner: {best} F1={results[best]['f1']:.4f}")

In [ ]:
# A/B results: comparison table and F1 bar plot
import matplotlib.pyplot as plt
import seaborn as sns

results_df = pd.DataFrame.from_dict(results, orient="index")
results_df = results_df.sort_values("f1", ascending=False)
results_df[["f1", "acc"]] = results_df[["f1", "acc"]].round(4)
results_df

In [ ]:
# Plot F1 scores per model
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 4))
sns.barplot(x=results_df.index, y="f1", data=results_df, palette="viridis")
plt.ylabel("F1 (weighted)")
plt.xlabel("Model")
plt.title("TruthLens UA — A/B Test F1 Comparison")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# 3.x Висновки та рекомендації щодо моделей

- **Лідер за F1 (winner)**: модель `best` з попередньої клітинки (очікувано — `LinearSVC_C1` на ISOT+UA).
- **LinearSVC** зазвичай дає найкращий баланс між якістю та швидкістю, особливо на великих tf-idf ознаках.
- **LogisticRegression** може бути цікавою альтернативою, якщо потрібні ймовірності (`predict_proba`).
- **RandomForest** зазвичай повільніший і гірше масштабується на 50k tf-idf фіч, тому його варто залишити як бенчмарк.
- **Рекомендація для продакшену TruthLens UA**: використовувати `LinearSVC` як основну модель, а результати A/B тестів (ця таблиця й графік) включити в розділ 3 диплому як доказ вибору моделі.